In [1]:
from langchain_core.documents import Document

In [2]:
doc=Document(
    page_content="this is the main text that will be used for the rag",
    metadata={
        "source":"dummy.txt",
        "pages":10,
        "author":"Sohan",
        
    }
)

In [3]:
doc

Document(metadata={'source': 'dummy.txt', 'pages': 10, 'author': 'Sohan'}, page_content='this is the main text that will be used for the rag')

In [4]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("created and written to files")

created and written to files


In [6]:
from langchain_community.document_loaders import TextLoader

c:\Users\LENONO\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")

In [8]:
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [9]:
from langchain_community.document_loaders import DirectoryLoader

In [10]:
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    show_progress=True,
    loader_cls=TextLoader
)

In [11]:
documents=dir_loader.load()
documents[0]

100%|██████████| 2/2 [00:00<00:00, 119.51it/s]


Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n    \n    \n    ')

In [12]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

In [13]:
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

In [14]:
pdf_docs=dir_loader.load()
pdf_docs

100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5\nExplain thread and its advantages.\n6\nExplain pthread_create() and pthread_join().\n7\nWhat is mutex? Why is it used?\n8\nDefine semaphore and list its

In [15]:
from pathlib import Path

In [ ]:
def process_all_pdf_files(pdf_directory):
    all_docs = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} files to process")

    for pdf_file in pdf_files:
        try:
            print(f"loading : {pdf_file.name}")
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load() #this returns a list

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_docs.extend(documents)  

        except Exception as e:
            print(f"error : {e}")

    print(f"total number of documents : {len(all_docs)}")
    return all_docs

In [ ]:
print(type(documents))

<class 'list'>


In [17]:
all_pdf_documents=process_all_pdf_files("../data")

found 1 files to process
loading : LOS_Question_Bank.pdf
total number of documents : 1


Splitting the text into chunks

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
def split_documents(documents,chunk_size=100,chunk_overlap=20):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content : {split_docs[0].page_content[:200]}")
        print(f"Metadata : {split_docs[0].metadata}")
    return split_docs

In [20]:
chunks=split_documents(all_pdf_documents)
chunks

split 1 documents into 16 chunks
Content : Linux Operating Systems (LOS)
Question Bank – 5 & 10 Marks
5 MARK QUESTIONS
1
Metadata : {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonym

In [41]:
print(type(chunks))
chunks[0]

<class 'list'>


Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1')

Embedding store and vectorDB

In [21]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully")
        except Exception as e:
            print(f"failed to load model : {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with dimensions : {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1236.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded successfully


Here we store in the VectorDB

In [ ]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persistent_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.collection=None
        self.client=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_dir)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desc":"pdf embeddings for RAG"}
            )
            print(f"vector store initialized :{self.collection_name}")
            print(f"existing documents in the collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must be equal to the number of embeddings")
        
        print(f"adding {len(documents)} documents to the vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"added {len(documents)} documents to the vector store")
        
        except Exception as e:
            print(f"error storing the documents into the vector store : {e}")
            raise

vectorstore=VectorStore()

vector store initialized :pdf_documents
existing documents in the collection : 32


In [24]:
vectorstore

In [25]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonym

In [26]:
texts=[doc.page_content for doc in chunks]
embeddings=embedding_manager.generate_embeddings(texts)
vectorstore.add_documents(chunks,embeddings)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Generated embeddings with dimensions : (16, 384)
adding 16 documents to the vector store
add 16 documents to the vector store


Retriever pipeline from the VectorStore

In [27]:
class RAGretriever:
    def __init__(self, vector_store:VectorStore,embedding_manager:EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int = 5,score_threshold:float = 0.0)->list[Dict[str,Any]]: #the query is the question, the top_k is the top 5 results in this case and the threshold is the minimum similarity threshold, like min they should be like 0 similar
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate (zip(ids,documents,metadatas,distances)):
                    similarity_score=1-distance

                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"retrieved {len(retrieved_docs)} documents")
            else:
                print("No documents found")
            return retrieved_docs
        
        except Exception as e:
            print(f"error occurred : {e}")
            return []



rag_retriever=RAGretriever(vectorstore,embedding_manager)


In [28]:
rag_retriever

In [29]:

rag_retriever.retrieve("what is signal in linux")

Batches: 100%|██████████| 1/1 [00:00<00:00, 29.35it/s]

Generated embeddings with dimensions : (1, 384)
retrieved 5 documents


[{'id': 'doc_312cc095_9',
  'content': '10 MARK QUESTIONS\n1\nExplain signal handling in Linux using signal() and sigaction().\n2',
  'metadata': {'title': '(anonymous)',
   'creationDate': "D:20251230134532+00'00'",
   'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf',
   'subject': '(unspecified)',
   'format': 'PDF 1.4',
   'keywords': '',
   'moddate': '2025-12-30T13:45:32+00:00',
   'doc_index': 9,
   'source': '..\\data\\pdf\\LOS_Question_Bank.pdf',
   'total_pages': 1,
   'file_type': 'pdf',
   'trapped': '',
   'modDate': "D:20251230134532+00'00'",
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'content_length': 86,
   'creationdate': '2025-12-30T13:45:32+00:00',
   'creator': '(unspecified)',
   'page': 0,
   'source_file': 'LOS_Question_Bank.pdf',
   'author': '(anonymous)'},
  'similarity_score': 0.5268689692020416,
  'distance': 0.4731310307979584,
  'rank': 1},
 {'id': 'doc_ca25ea9a_9',
  'content': '10 MARK QUESTIONS\n1\nExplain signal handling in Linux 

In [30]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0, 'source_file': 'LOS_Question_Bank.pdf', 'file_type': 'pdf'}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5\nExplain thread and its advantages.\n6\nExplain pthread_create() and pthread_join().\n7\nWhat

In [31]:
print(type(all_pdf_documents))
print(type(all_pdf_documents[0]))
print(all_pdf_documents[0])

<class 'list'>
<class 'langchain_core.documents.base.Document'>
page_content='Linux Operating Systems (LOS)
Question Bank – 5 & 10 Marks
5 MARK QUESTIONS
1
Define a signal in Linux. List any four commonly used signals.
2
Explain the purpose of signal() system call.
3
What is sigaction()? How is it better than signal()?
4
Differentiate between kill() and raise().
5
Explain thread and its advantages.
6
Explain pthread_create() and pthread_join().
7
What is mutex? Why is it used?
8
Define semaphore and list its types.
9
Explain pipe() IPC mechanism.
10 What is shared memory?
11 Define socket.
12 List socket API functions.
13 What is gdb?
14 Explain gprof.
15 What is valgrind?
10 MARK QUESTIONS
1
Explain signal handling in Linux using signal() and sigaction().
2
Explain sending and receiving signals using kill() and raise().
3
Explain thread creation and synchronization using mutex.
4
Explain semaphores with example.
5
Explain pipes as IPC mechanism.
6
Explain message queues with example.


Now we integrate the VectorDB context pipeline with the LLM Output

In [32]:
import ollama

In [36]:

import google.generativeai as genai

genai.configure(api_key="")

model = genai.GenerativeModel("gemini-2.5-flash")

In [37]:
def generate_answer(query,retrieved_docs):
    context="\n\n".join([doc['content'] for doc in retrieved_docs])

    prompt=f"""
    Answer the question using the context below

    Context: {context}

    Question" {query}
    """
    response=model.generate_content(prompt)
    if not response:
        raise ValueError("Ollama failed to load")

    return response.text

In [38]:
retrieved=rag_retriever.retrieve("what is signal in linux")
answer=generate_answer("what is signal in linux",retrieved)

print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.62it/s]


Generated embeddings with dimensions : (1, 384)
retrieved 5 documents
The provided context asks to "Define a signal in Linux" as a 5-mark question, but it does not provide the definition itself.
